In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

# (a)
# Load ratings file
ratings = pd.read_csv("https://drive.google.com/file/d/1itbJEyYRw45Rq3xwl8sfJeHcmA2xwrKP/view?usp=sharing")

# Show first few rows
print("\n✅ First five rows:")
print(ratings.head())

# Compute dataset summary
num_ratings = len(ratings)
num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()

print(f"\n📊 Dataset Summary:")
print(f"Row Count for Ratings Dataset: {num_ratings}")
print(f"Number of Users: {num_users}")
print(f"Number of Movies: {num_movies}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/Recommonder_System/ratings.csv'

In [ ]:
# Create the user–item matrix
# Create pivot table: rows = userId, columns = movieId, values = rating
user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')

# Show a small sample: first 10 users and first 10 movies
display(user_movie_matrix.iloc[:10, :10])

movieId,1,2,3,4,5,6,7,8,9,10
userId,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0
7,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# (b) Define manual Pearson correlation similarity
def pearson_similarity(user_a, user_b):

    # Find common movies rated by both users
    common_items = user_movie_matrix.loc[user_a].notna() & user_movie_matrix.loc[user_b].notna()

    if common_items.sum() == 0:
        return 0.0  # No overlap in ratings

    # Extract their ratings for common movies
    a_ratings = user_movie_matrix.loc[user_a, common_items]
    b_ratings = user_movie_matrix.loc[user_b, common_items]

    # Compute means
    a_mean = a_ratings.mean()
    b_mean = b_ratings.mean()

    # Apply formula
    numerator = ((a_ratings - a_mean) * (b_ratings - b_mean)).sum()
    denominator = np.sqrt(((a_ratings - a_mean)**2).sum()) * np.sqrt(((b_ratings - b_mean)**2).sum())

    if denominator == 0:
        return 0.0

    return numerator / denominator

# Compute full user-user similarity matrix
def compute_user_similarity_matrix(user_item_matrix):
    users = user_movie_matrix.index
    similarity_matrix = pd.DataFrame(index=users, columns=users, dtype=float)

    for u1 in users:
        for u2 in users:
            if u1 == u2:
                similarity_matrix.loc[u1, u2] = 1.0
            else:
                similarity_matrix.loc[u1, u2] = pearson_similarity(u1, u2)

    return similarity_matrix.fillna(0)


# Compute the similarity matrix
similarity_matrix = compute_user_similarity_matrix(user_movie_matrix)

# Show a sample (10x10)
print("\n✅ Pearson User-Based Similarity Matrix (Top 10×10 section):")
print(similarity_matrix.iloc[:10, :10])


✅ Pearson User-Based Similarity Matrix (Top 10×10 section):
userId        1         2         3         4         5         6         7   \
userId                                                                         
1       1.000000  0.000000  0.079819  0.207983  0.268749 -0.291636 -0.118773   
2       0.000000  1.000000  0.000000  0.000000  0.000000  0.000000 -0.991241   
3       0.079819  0.000000  1.000000  0.000000  0.000000  0.000000  0.000000   
4       0.207983  0.000000  0.000000  1.000000 -0.336525  0.148498  0.542861   
5       0.268749  0.000000  0.000000 -0.336525  1.000000  0.043166  0.158114   
6      -0.291636  0.000000  0.000000  0.148498  0.043166  1.000000 -0.126595   
7      -0.118773 -0.991241  0.000000  0.542861  0.158114 -0.126595  1.000000   
8       0.469668  0.000000  0.000000  0.117851  0.028347 -0.200062  0.220416   
9       0.918559  0.000000  0.000000  0.000000  0.000000  0.000000  0.925000   
10     -0.037987  0.037796  0.000000  0.485794 -0.777714  0

In [ ]:
# (c)
# Function to predict a missing rating using Pearson similarity
def predict_rating(target_user, target_movie, user_movie_matrix, similarity_matrix):

    # Users who rated this movie
    users_who_rated = user_movie_matrix[target_movie].dropna()

    # If nobody rated the movie, cannot predict
    if users_who_rated.empty:
        return np.nan

    # Similarities between target_user and the users who rated this movie
    sims = similarity_matrix.loc[target_user, users_who_rated.index]

    # Ratings by those users
    ratings = users_who_rated

    # Compute each r_i - mean_i for weighted average
    mean_ratings = user_movie_matrix.loc[users_who_rated.index].mean(axis=1)
    rating_deviation = ratings - mean_ratings

    # Weighted sum of deviations
    numerator = np.sum(sims * rating_deviation)
    denominator = np.sum(np.abs(sims))

    # Avoid division by zero
    if denominator == 0 or np.isnan(denominator):
        return np.nan

    # Add back target user's mean to get the final prediction
    user_mean = user_movie_matrix.loc[target_user].mean()
    return user_mean + numerator / denominator


# Try an example prediction ---
example_user = 1
# find a movie that user 1 has NOT rated
unrated_movies = user_movie_matrix.loc[example_user][user_movie_matrix.loc[example_user].isna()].index

# pick one unrated movie for testing
if len(unrated_movies) > 0:
    example_movie = unrated_movies[0]
    predicted_value = predict_rating(example_user, example_movie, user_movie_matrix, similarity_matrix)
    print(f"Predicted rating for user {example_user} on movie {example_movie}: {predicted_value:.2f}")
else:
    print(f"User {example_user} has rated all movies!")

Predicted rating for user 1 on movie 2: 4.16


In [ ]:
# (extended): Predict for users 1–4

target_users = [1, 2, 3, 4]
prediction_summary = []

for target_user in target_users:
    # Movies that this user hasn't rated
    unrated_movies = user_movie_matrix.loc[target_user][user_movie_matrix.loc[target_user].isna()].index

    for movie in unrated_movies[:15]:  # limit to 15 per user for clarity
        pred = predict_rating(target_user, movie, user_movie_matrix, similarity_matrix)

        if not np.isnan(pred):
            # Find the most similar user who rated this movie
            users_who_rated = user_movie_matrix[movie].dropna().index
            similarities = similarity_matrix.loc[target_user, users_who_rated]

            most_similar_user = similarities.idxmax()
            highest_similarity = similarities.max()

            prediction_summary.append({
                "User_Id": target_user,
                "Movie_Id": movie,
                "Predicted_Rating": round(pred, 2),
                "Most_Similar_User": most_similar_user,
                "Similarity": round(highest_similarity, 4)
            })

# Convert to DataFrame
pred_summary_df = pd.DataFrame(prediction_summary)

# Display predictions
print("\n✅ Predicted Ratings Summary for first 10 users:")
display(pred_summary_df.head(20))


✅ Predicted Ratings Summary for first 10 users:


,User_Id,Movie_Id,Predicted_Rating,Most_Similar_User,Similarity
0,1,2,4.16,476,0.7869
1,1,4,3.74,162,0.7083
2,1,5,4.11,120,0.5949
3,1,7,4.24,90,0.8216
4,1,8,3.98,20,0.4196
5,1,9,3.72,270,0.4663
6,1,10,4.24,476,0.7869
7,1,11,4.39,476,0.7869
8,1,12,4.00,44,0.6844
9,1,13,4.23,476,0.7869


In [ ]:
# (d)
# New similarity function

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# X: user-item rating matrix, rows=users, columns=movies
def vectorized_user_similarity(X):
    # Center ratings by user mean
    user_means = np.nanmean(X, axis=1, keepdims=True)
    X_centered = X - user_means

    # Replace NaNs with 0 to compute cosine similarity
    X_filled = np.nan_to_num(X_centered, nan=0.0)

    # Compute user-user similarity matrix
    return cosine_similarity(X_filled)

# Convert user-item DataFrame to numpy array
X = user_movie_matrix.values  # shape: (num_users, num_movies)

# Compute vectorized similarity
user_similarity_matrix = vectorized_user_similarity(X)

# Wrap into DataFrame for convenience (optional)
user_ids = user_movie_matrix.index
similarity_df = pd.DataFrame(user_similarity_matrix, index=user_ids, columns=user_ids)

print("\n✅ Vectorized Adjusted Cosine Similarity of first 10 users:")
print(similarity_df.iloc[:10, :10])


✅ Vectorized Adjusted Cosine Similarity of first 10 users:
userId        1         2         3         4         5         6         7   \
userId                                                                         
1       1.000000  0.001265  0.000553  0.048419  0.021847 -0.045497 -0.006200   
2       0.001265  1.000000  0.000000 -0.017164  0.021796 -0.021051 -0.011114   
3       0.000553  0.000000  1.000000 -0.011260 -0.031539  0.004800  0.000000   
4       0.048419 -0.017164 -0.011260  1.000000 -0.029620  0.013956  0.058091   
5       0.021847  0.021796 -0.031539 -0.029620  1.000000  0.009111  0.010117   
6      -0.045497 -0.021051  0.004800  0.013956  0.009111  1.000000  0.004709   
7      -0.006200 -0.011114  0.000000  0.058091  0.010117  0.004709  1.000000   
8       0.047013 -0.048085 -0.032471  0.002065 -0.012284 -0.075888  0.031098   
9       0.019510  0.000000  0.000000 -0.005874  0.000000 -0.000026  0.064350   
10     -0.008754  0.003012  0.000000  0.051590 -0.033165  0.

In [ ]:
# Predict rating for a single (user, movie) pair
def predict(user_idx, movie_idx, X, users_similarity):
    user_mean = np.nanmean(X[user_idx])
    users_who_rated = np.where(~np.isnan(X[:, movie_idx]))[0]

    weighted_sum = 0.0
    similarity_sum = 0.0

    for neighbor in users_who_rated:
        if neighbor == user_idx:
            continue
        sim = users_similarity[user_idx, neighbor]
        if sim == 0 or np.isnan(sim):
            continue
        neighbor_mean = np.nanmean(X[neighbor])
        neighbor_rating = X[neighbor, movie_idx]
        weighted_sum += sim * (neighbor_rating - neighbor_mean)
        similarity_sum += abs(sim)

    if similarity_sum == 0:
        return user_mean

    pred = user_mean + (weighted_sum / similarity_sum)

    # Optional: clip slightly within the rating scale (e.g., 0.5–5)
    return min(max(pred, 0.5), 5)

In [ ]:
target_users = [1, 2, 3, 4]
prediction_summary = []

# Convert DataFrame to numpy for faster access
X = user_movie_matrix.values
similarity = user_similarity_matrix  # from Step 8

user_index_map = {uid: i for i, uid in enumerate(user_movie_matrix.index)}
movie_index_map = {mid: j for j, mid in enumerate(user_movie_matrix.columns)}

for u in target_users:
    u_idx = user_index_map[u]

    # Movies that user has not rated
    unrated_movies = user_movie_matrix.loc[u][user_movie_matrix.loc[u].isna()].index

    for movie in unrated_movies[:15]:  # limit to 15 for clarity
        m_idx = movie_index_map[movie]
        pred = predict(u_idx, m_idx, X, similarity)

        # Get users who rated this movie
        users_who_rated = user_movie_matrix[movie].dropna().index

        # Find most similar user among raters
        sim_values = [(other_u, similarity[user_index_map[u], user_index_map[other_u]])
                      for other_u in users_who_rated if other_u != u]

        if len(sim_values) == 0:
            continue

        most_similar_user, highest_sim = max(sim_values, key=lambda x: x[1])

        prediction_summary.append({
            "User_Id": u,
            "Movie_Id": movie,
            "New_Predicted_Rating": round(pred, 2),
            "Most_Similar_User": most_similar_user,
            "New_Similarity": round(highest_sim, 4)
        })

pred_summary_df = pd.DataFrame(prediction_summary)

print("\n✅ Predicted Ratings Summary for first 10 users:")
display(pred_summary_df.head(20))


✅ Predicted Ratings Summary for first 10 users:


,User_Id,Movie_Id,New_Predicted_Rating,Most_Similar_User,New_Similarity
0,1,2,4.11,414,0.1013
1,1,4,4.07,600,0.0695
2,1,5,4.07,414,0.1013
3,1,7,4.08,597,0.1026
4,1,8,3.90,414,0.1013
5,1,9,4.08,464,0.0608
6,1,10,4.31,301,0.1248
7,1,11,4.31,597,0.1026
8,1,12,3.74,120,0.0928
9,1,13,4.28,19,0.0852


In [ ]:
# (e)
# Convert user-item DataFrame to numpy array
X = user_movie_matrix.values

# Compute vectorized adjusted cosine similarity (fast)
from sklearn.metrics.pairwise import cosine_similarity

def vectorized_user_similarity(X):
    user_means = np.nanmean(X, axis=1, keepdims=True)
    X_centered = X - user_means
    X_filled = np.nan_to_num(X_centered, nan=0.0)
    return cosine_similarity(X_filled)

users_similarity = vectorized_user_similarity(X)

In [ ]:
def predict_rating(user_idx, movie_idx, X, users_similarity):
    user_mean = np.nanmean(X[user_idx])
    users_who_rated = np.where(~np.isnan(X[:, movie_idx]))[0]

    weighted_sum = 0.0
    similarity_sum = 0.0

    for neighbor in users_who_rated:
        if neighbor == user_idx:
            continue
        sim = users_similarity[user_idx, neighbor]
        if sim == 0 or np.isnan(sim):
            continue
        neighbor_mean = np.nanmean(X[neighbor])
        neighbor_rating = X[neighbor, movie_idx]

        weighted_sum += sim * (neighbor_rating - neighbor_mean)
        similarity_sum += abs(sim)

    if similarity_sum == 0:
        return user_mean

    return np.clip(user_mean + (weighted_sum / similarity_sum), 0, 5)


In [ ]:
def get_top_recommendations(user_idx, X, users_similarity, n=None):
    unrated_movies = np.where(np.isnan(X[user_idx]))[0]
    preds = {}

    for movie_idx in unrated_movies:
        pred = predict_rating(user_idx, movie_idx, X, users_similarity)
        preds[movie_idx] = pred

    # Convert to sorted DataFrame
    preds_df = pd.DataFrame(list(preds.items()), columns=["movie_idx", "predicted_rating"])
    return preds_df.sort_values("predicted_rating", ascending=False).head(n)

In [ ]:
def group_member_recommendations(group_users, X, users_similarity, n=None):
    user_recs = {}
    for u in group_users:
        user_recs[u] = get_top_recommendations(u, X, users_similarity, n)
    return user_recs

In [ ]:
def aggregate_group_recommendations(user_recs, method="average"):
    import pandas as pd

    all_recs = pd.concat(user_recs.values(), keys=user_recs.keys(), names=["userId"])
    all_recs = all_recs.reset_index(level=0)

    if method == "average":
        group_scores = all_recs.groupby("movie_idx")["predicted_rating"].mean()
    elif method == "least_misery":
        group_scores = all_recs.groupby("movie_idx")["predicted_rating"].min()
    else:
        raise ValueError("Invalid aggregation method")

    group_recs = group_scores.sort_values(ascending=False).reset_index()
    group_recs.columns = ["movie_idx", "group_score"]
    return group_recs

In [ ]:
group_users = list(range(5))
user_recs = group_member_recommendations(group_users, X, users_similarity, n= None)

avg_group_recs = aggregate_group_recommendations(user_recs, method="average")
lm_group_recs = aggregate_group_recommendations(user_recs, method="least_misery")

print("Group Recommendations (Average Method):")
display(avg_group_recs.head(10))

print("Group Recommendations (Least Misery Method):")
display(lm_group_recs.head(10))

Group Recommendations (Average Method):


,movie_idx,group_score
0,7934,4.720832
1,9094,4.679147
2,9703,4.679147
3,3754,4.660447
4,5572,4.660447
5,5862,4.660447
6,7603,4.641390
7,5822,4.596677
8,1025,4.588172
9,3411,4.588172


Group Recommendations (Least Misery Method):


,movie_idx,group_score
0,7603,4.357954
1,9066,4.183035
2,8054,4.183035
3,961,3.829261
4,1543,3.811976
5,1492,3.759177
6,5511,3.742217
7,6100,3.736568
8,5525,3.736568
9,7593,3.736568


In [ ]:
# (f)
def aggregate_group_recommendations_disagreement(user_recs, lambda_penalty=0.5):
    import pandas as pd

    # Combine all user recommendation lists
    all_recs = pd.concat(user_recs.values(), keys=user_recs.keys(), names=["userId"])
    all_recs = all_recs.reset_index(level=0)

    # Compute average rating per movie
    avg_scores = all_recs.groupby("movie_idx")["predicted_rating"].mean()

    # Compute disagreement (standard deviation) per movie
    disagreement = all_recs.groupby("movie_idx")["predicted_rating"].std()

    # Final group score = mean - penalty * disagreement
    group_score = avg_scores - lambda_penalty * disagreement

    # Combine results into one DataFrame
    result = pd.DataFrame({
        "movie_idx": avg_scores.index,
        "avg_score": avg_scores.values,
        "disagreement": disagreement.values,
        "group_score": group_score.values
    })

    # Sort best recommendations first
    result = result.sort_values(by="group_score", ascending=False).reset_index(drop=True)

    return result

In [ ]:
dis_group_recs = aggregate_group_recommendations_disagreement(user_recs, lambda_penalty=0.5)

print("✅ Group Recommendations (Disagreement-Aware Method)")
display(dis_group_recs.head(10))

✅ Group Recommendations (Disagreement-Aware Method)


,movie_idx,avg_score,disagreement,group_score
0,7603,4.641390,0.328606,4.477087
1,7934,4.720832,0.570906,4.435379
2,9094,4.679147,0.557580,4.400357
3,9703,4.679147,0.557580,4.400357
4,5572,4.660447,0.566734,4.377080
5,5862,4.660447,0.566734,4.377080
6,3754,4.660447,0.566734,4.377080
7,5822,4.596677,0.559724,4.316814
8,1025,4.588172,0.605345,4.285500
9,3411,4.588172,0.605345,4.285500
